# Filtro de Spam — Sistema Especialista (A1)

**UDF · Inteligência Artificial e LFA · Profa. Kadidja Valéria**
**Grupo 13:** Lucas Dórea, Nicolas Angelo, Deivid Cerqueira
**Repositório:** https://github.com/Lucasdoreac/filtro-spam-lfa

Roteiro do notebook:
1. Descrição do SE
2. Base de conhecimento e regras
3. Motor de inferência (encadeamento para trás), com rastro
4. Casos de teste: positivo, negativo e informação ausente
5. Comparação com o encadeamento para frente
6. Testes automatizados
7. Arquitetura híbrida: árvore de decisão + regras (dados **fictícios**)
8. Ética e limitações
9. Cola da prova (resumo para consulta)

> Rode as células em ordem (**Ambiente de execução → Executar tudo**).

## 0. Preparação

Quando o notebook roda no Colab, a célula abaixo clona o repositório e usa **os mesmos arquivos** `base.py`, `motor.py` e `cli.py` do projeto. Nada de código é duplicado aqui.

In [1]:
import os, sys, subprocess

if not os.path.exists("motor.py"):
    if not os.path.exists("filtro-spam-lfa"):
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/Lucasdoreac/filtro-spam-lfa"], check=True)
    os.chdir("filtro-spam-lfa")
sys.path.insert(0, os.getcwd())
print("Pasta do projeto:", os.path.basename(os.getcwd()))

Pasta do projeto: repo


## 1. Descrição do SE

- **Finalidade:** ajudar uma pessoa a decidir se uma mensagem (e-mail, SMS ou WhatsApp) merece **revisão cuidadosa** antes de clicar ou responder.
- **Tarefa:** diagnóstico e interpretação, isto é, explicar a partir de evidências.
- **Fatos de entrada:** 4 perguntas respondidas com `s` (sim), `n` (não) ou `?` (não sei).
- **Objetivo:** `revisao_recomendada`.
- **Resultado esperado em um caso:** remetente desconhecido = sim e link duvidoso = sim → R1 deriva **spam** → R3 deriva **revisão humana recomendada**.
- **Limite:** o sistema **nunca move nem apaga** mensagens. A decisão final é humana.

## 2. Base de conhecimento e regras (`base.py`)

In [2]:
from base import REGRAS, PERGUNTAS, OBJETIVO, Regra

print("Objetivo:", OBJETIVO, "\n")
print("Fatos perguntados ao usuário:")
for fato, pergunta in PERGUNTAS.items():
    print(f"  {fato:24} {pergunta}")

print("\nRegras (SE ... ENTÃO ...):")
for r in REGRAS:
    condicoes = " E ".join(f"{s} = {'sim' if v else 'não'}" for s, v in r.premissas)
    print(f"  {r.id}: SE {condicoes} ENTÃO {r.conclusao}   [origem: {r.origem}]")

Objetivo: revisao_recomendada 

Fatos perguntados ao usuário:
  remetente_desconhecido   O remetente é desconhecido (número ou endereço que você nunca viu)?
  link_encurtado           A mensagem traz link encurtado ou de origem duvidosa?
  promessa_ganho_facil     A mensagem promete prêmio, dinheiro ou ganho fácil?
  urgencia                 A mensagem pressiona por resposta imediata?

Regras (SE ... ENTÃO ...):
  R1: SE remetente_desconhecido = sim E link_encurtado = sim ENTÃO spam   [origem: hipótese didática, sem validação empírica]
  R2: SE promessa_ganho_facil = sim E urgencia = sim ENTÃO spam   [origem: hipótese didática, sem validação empírica]
  R3: SE spam = sim ENTÃO revisao_recomendada   [origem: hipótese didática, sem validação empírica]


- **R1 e R2** são caminhos alternativos (um OU outro) para concluir `spam`.
- **R3 encadeia**: usa a conclusão intermediária `spam` para chegar ao objetivo.
- **Fatos perguntáveis** (folhas) vêm do usuário. **Fatos derivados** (`spam`, `revisao_recomendada`) só são produzidos por regras.
- Base (`base.py`) e motor (`motor.py`) ficam **separados**: trocar regras não exige mexer no motor.

## 3. Motor de inferência (`motor.py`): encadeamento para trás

1. Parte do **objetivo** `revisao_recomendada`.
2. Procura as regras que **concluem** esse objetivo (R3).
3. Cada premissa vira um **subobjetivo** (`spam`, que leva a R1 e R2).
4. Fato que nenhuma regra conclui é **perguntado** ao usuário, uma única vez, e guardado na memória de trabalho (`Motor.fatos`).
5. Premissa falsa → regra **refutada**, e as demais premissas nem são perguntadas. Premissa `?` → regra **inconclusiva**. Todas verdadeiras → regra **confirmada**.
6. Ciclo entre regras termina como inconclusivo. Em caso de conflito, vale a ordem fixa das regras.

A função abaixo roda o motor com respostas prontas e mostra **quais perguntas foram feitas** e o **rastro** gerado por `cli.explicar`.

In [3]:
from motor import Motor
from cli import explicar

TEXTO = {True: "s", False: "n", None: "?"}

def rodar(nome, respostas, regras=REGRAS, objetivo=OBJETIVO):
    motor = Motor(regras, lambda simbolo: respostas[simbolo])
    resultado = motor.consultar(objetivo)
    print(f"=== {nome} ===")
    print("Perguntas feitas:", ", ".join(f"{p}={TEXTO[respostas[p]]}" for p in resultado.perguntados))
    for linha in explicar(resultado):
        print(linha)
    print()
    return resultado

## 4. Casos de teste (positivo, negativo e informação ausente)

In [4]:
caso_a = rodar("A · positivo via R1", {
    "remetente_desconhecido": True, "link_encurtado": True})

caso_b = rodar("B · positivo via R2 (R1 refutada)", {
    "remetente_desconhecido": False, "promessa_ganho_facil": True, "urgencia": True})

caso_c = rodar("C · negativo", {
    "remetente_desconhecido": False, "promessa_ganho_facil": False})

caso_d = rodar("D · informação ausente", {
    "remetente_desconhecido": None, "link_encurtado": True, "promessa_ganho_facil": False})

=== A · positivo via R1 ===
Perguntas feitas: remetente_desconhecido=s, link_encurtado=s
Conclusão: características compatíveis com spam.
Ação recomendada: revisar a mensagem antes de decidir o que fazer.
Nenhuma mensagem foi movida ou apagada. A decisão final é humana.
Regras verificadas:
  R1 confirmada: remetente_desconhecido = sim, link_encurtado = sim
  R3 confirmada: spam = sim
Limitação: regras são hipóteses didáticas, sem validação empírica.

=== B · positivo via R2 (R1 refutada) ===
Perguntas feitas: remetente_desconhecido=n, promessa_ganho_facil=s, urgencia=s
Conclusão: características compatíveis com spam.
Ação recomendada: revisar a mensagem antes de decidir o que fazer.
Nenhuma mensagem foi movida ou apagada. A decisão final é humana.
Regras verificadas:
  R1 refutada: remetente_desconhecido = não
  R2 confirmada: promessa_ganho_facil = sim, urgencia = sim
  R3 confirmada: spam = sim
Limitação: regras são hipóteses didáticas, sem validação empírica.

=== C · negativo ===
P

**Leitura dos resultados**
- **A:** bastaram **2 perguntas**. Com R1 confirmada, o objetivo já está provado, então ganho e urgência nem são perguntados.
- **B:** R1 é refutada logo na 1ª premissa (o link nem é perguntado), e R2 confirma.
- **C:** nenhuma regra confirmou, o que **não significa mensagem segura** (pode ser falso negativo).
- **D:** `?` **não vira "não"**. R1 fica inconclusiva e o sistema informa o que falta.

**Como?** O caso A veio de **R3**, apoiada por **R1**. **Por quê?** O link é perguntado porque **R1 precisa dessa premissa**.

## 5. Comparação: encadeamento para frente

Para comparar, a célula abaixo aplica **as mesmas regras** de forma **dirigida por dados**: parte de todos os fatos e dispara as regras até nada novo surgir. Essa função é só demonstrativa; o projeto usa o motor para trás.

In [5]:
def encadear_para_frente(regras, fatos):
    fatos = dict(fatos)
    ciclo = 0
    while True:
        ciclo += 1
        disparou = False
        for r in regras:
            if fatos.get(r.conclusao):
                continue
            if all(fatos.get(s) is v for s, v in r.premissas):
                fatos[r.conclusao] = True
                print(f"ciclo {ciclo}: {r.id} disparou -> {r.conclusao}")
                disparou = True
        if not disparou:
            print(f"ciclo {ciclo}: nenhuma regra nova -> parada")
            return fatos

todos = {"remetente_desconhecido": True, "link_encurtado": True,
         "promessa_ganho_facil": False, "urgencia": False}
final = encadear_para_frente(REGRAS, todos)
print("revisao_recomendada =", final.get("revisao_recomendada", False))

ciclo 1: R1 disparou -> spam
ciclo 1: R3 disparou -> revisao_recomendada
ciclo 2: nenhuma regra nova -> parada
revisao_recomendada = True


| | Para frente | Para trás (o nosso) |
|---|---|---|
| Parte de | fatos disponíveis | objetivo / hipótese |
| Pergunta-guia | que conclusões os fatos permitem? | que evidências sustentam a hipótese? |
| No caso A | precisa dos **4** fatos antes de começar | faz só **2** perguntas |

Escolhemos **para trás** porque há um objetivo único e o motor pergunta apenas o necessário.

## 6. Testes automatizados (`test_motor.py`)

In [6]:
!python -m unittest discover -v 2>&1 | tail -25

test_tres_regras_com_origem_declarada (test_motor.TesteBase.test_tres_regras_com_origem_declarada) ... ok
test_aceita_espacos_e_caixa (test_motor.TesteCli.test_aceita_espacos_e_caixa) ... ok
test_consulta_completa_confirmada (test_motor.TesteCli.test_consulta_completa_confirmada) ... ok
test_consulta_sem_informacao_suficiente (test_motor.TesteCli.test_consulta_sem_informacao_suficiente) ... ok
test_entrada_invalida_pede_de_novo (test_motor.TesteCli.test_entrada_invalida_pede_de_novo) ... ok
test_ausencia_de_sinais_nao_prova_seguranca (test_motor.TesteMotor.test_ausencia_de_sinais_nao_prova_seguranca) ... ok
test_caso_legitimo_com_link_pode_ser_falso_positivo (test_motor.TesteMotor.test_caso_legitimo_com_link_pode_ser_falso_positivo) ... ok
test_ciclo_termina_como_inconclusivo (test_motor.TesteMotor.test_ciclo_termina_como_inconclusivo) ... ok
test_desconhecido_nao_vira_nao (test_motor.TesteMotor.test_desconhecido_nao_vira_nao) ... ok
test_fato_compartilhado_e_perguntado_uma_vez (test_m

- **Positivo:** `test_r1_confirma_e_encadeia_r3`
- **Negativo:** `test_premissa_falsa_nao_pergunta_o_resto_e_r3_refuta`
- **Informação ausente:** `test_desconhecido_nao_vira_nao`
- **Verificação** (a lógica está certa: ciclos, premissas) é coberta pelos testes. **Validação** com mensagens reais **não foi feita**.

## 7. Arquitetura híbrida: árvore de decisão + regras

**Fase 1 (aprendizado supervisionado):** uma árvore de decisão aprende a prever `suspeita_ml` (sim/não) a partir de atributos da mensagem.
**Fase 2 (sistema especialista):** a previsão vira um **fato gerado dinamicamente**, usado por uma regra nova:

> **R4:** SE `suspeita_ml` = sim E `remetente_desconhecido` = sim ENTÃO `spam`

⚠️ **Os dados abaixo são fictícios**, escritos pela equipe só para demonstrar o método. Eles **não representam** mensagens reais, e o resultado **não mede** a eficácia do filtro.

In [7]:
import pandas as pd

dados = pd.DataFrame([
    # mensagem fictícia,                               links, urgência, ganho, rótulo
    ("Reunião do grupo amanhã às 19h",                      0, 0, 0, "nao"),
    ("Segue o PDF da aula 12",                              1, 0, 0, "nao"),
    ("Sua fatura de novembro está disponível",              1, 0, 0, "nao"),
    ("Lembrete: prova A1 hoje",                             0, 1, 0, "nao"),
    ("Confirma o almoço de sábado?",                        0, 0, 0, "nao"),
    ("Link do formulário da disciplina",                    1, 0, 0, "nao"),
    ("Entrega do trabalho até sexta, não esqueça",          0, 1, 0, "nao"),
    ("Fotos da viagem no álbum compartilhado",              1, 0, 0, "nao"),
    ("Novo horário da monitoria",                           0, 0, 0, "nao"),
    ("Pix da conta de luz enviado",                         0, 0, 0, "nao"),
    ("Cupom de 10% na loja do campus",                      1, 0, 1, "nao"),
    ("Atualização do sistema acadêmico, veja o link",       1, 1, 0, "nao"),
    ("PARABÉNS! Você ganhou um iPhone, clique já",          2, 2, 2, "sim"),
    ("Sua conta será bloqueada hoje, confirme aqui",        2, 3, 0, "sim"),
    ("Ganhe R$ 5.000 por dia sem sair de casa",             1, 1, 3, "sim"),
    ("Último aviso: resgate seu prêmio agora",              2, 3, 2, "sim"),
    ("Herança à sua espera, responda urgente",              1, 2, 3, "sim"),
    ("Seu pacote está retido, pague a taxa no link",        3, 2, 0, "sim"),
    ("Investimento garantido 300%, só hoje",                2, 2, 3, "sim"),
    ("Clique e libere seu bônus antes que expire",          3, 2, 2, "sim"),
    ("Sorteio exclusivo: confirme seus dados",              2, 1, 2, "sim"),
    ("Empréstimo aprovado na hora, acesse",                 2, 2, 2, "sim"),
    ("Promoção relâmpago: 90% off nos links",               3, 2, 1, "sim"),
    ("Seu CPF tem pendência, regularize já",                2, 3, 0, "sim"),
], columns=["mensagem", "num_links", "termos_urgencia", "termos_ganho", "suspeita"])

ATRIBUTOS = ["num_links", "termos_urgencia", "termos_ganho"]
X = dados[ATRIBUTOS]          # atributos: descrevem o caso
y = dados["suspeita"]         # rótulo: resposta conhecida
print(dados["suspeita"].value_counts())
dados.head()

suspeita
nao    12
sim    12
Name: count, dtype: int64


,mensagem,num_links,termos_urgencia,termos_ganho,suspeita
0,Reunião do grupo amanhã às 19h,0,0,0,nao
1,Segue o PDF da aula 12,1,0,0,nao
2,Sua fatura de novembro está disponível,1,0,0,nao
3,Lembrete: prova A1 hoje,0,1,0,nao
4,Confirma o almoço de sábado?,0,0,0,nao


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import confusion_matrix, accuracy_score

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

modelo = DecisionTreeClassifier(max_depth=2, random_state=42)
modelo.fit(X_treino, y_treino)
previsoes = modelo.predict(X_teste)

print("Condições aprendidas pela árvore:")
print(export_text(modelo, feature_names=ATRIBUTOS))

Condições aprendidas pela árvore:
|--- termos_urgencia <= 0.50
|   |--- class: nao
|--- termos_urgencia >  0.50
|   |--- termos_urgencia <= 1.50
|   |   |--- class: nao
|   |--- termos_urgencia >  1.50
|   |   |--- class: sim



- `test_size=0.25`: reserva 25% dos exemplos para teste.
- `stratify=y`: mantém a proporção de sim e não nos dois conjuntos.
- `random_state=42`: torna a divisão reproduzível.
- `max_depth=2`: árvore curta e legível, com menos risco de decorar os dados.

In [9]:
rotulos = ["nao", "sim"]
matriz = pd.DataFrame(confusion_matrix(y_teste, previsoes, labels=rotulos),
                      index=[f"real {r}" for r in rotulos],
                      columns=[f"previsto {r}" for r in rotulos])
print(matriz, "\n")
print(f"Acurácia: {accuracy_score(y_teste, previsoes):.0%} em {len(y_teste)} casos de teste")

          previsto nao  previsto sim
real nao             3             0
real sim             1             2 

Acurácia: 83% em 6 casos de teste


⚠️ **Cautela:** com poucos casos de teste e dados fictícios, até 100% de acerto **não prova eficácia real**. A avaliação é instável, e o modelo pode errar fora do domínio em que foi treinado.

### Integração: a previsão vira fato para o motor

A regra R4 é acrescentada às regras originais. Quando o motor precisa do fato `suspeita_ml`, em vez de perguntar ao usuário, ele recebe a **previsão da árvore**.

In [10]:
R4 = Regra("R4", (("suspeita_ml", True), ("remetente_desconhecido", True)), "spam",
           "híbrida: previsão de árvore treinada com dados fictícios")
REGRAS_HIBRIDAS = REGRAS[:2] + (R4,) + REGRAS[2:]   # R1, R2, R4, R3

def rodar_hibrido(nome, atributos, respostas):
    previsao = modelo.predict(pd.DataFrame([atributos], columns=ATRIBUTOS))[0]
    fato_ml = previsao == "sim"
    print(f"Árvore: {atributos} -> suspeita_ml = {previsao}")
    def perguntar(simbolo):
        return fato_ml if simbolo == "suspeita_ml" else respostas[simbolo]
    resultado = Motor(REGRAS_HIBRIDAS, perguntar).consultar(OBJETIVO)
    print(f"=== {nome} ===")
    for linha in explicar(resultado):
        print(linha)
    print()
    return resultado

**Teste de três casos novos** (passo 3 da atividade da 2ª aula):

In [11]:
sinais_fracos = {"link_encurtado": False, "promessa_ganho_facil": False, "urgencia": False}

h1 = rodar_hibrido("H1 · árvore suspeita + remetente desconhecido",
                   {"num_links": 2, "termos_urgencia": 3, "termos_ganho": 0},
                   {"remetente_desconhecido": True, **sinais_fracos})

h2 = rodar_hibrido("H2 · árvore não suspeita + remetente desconhecido",
                   {"num_links": 0, "termos_urgencia": 0, "termos_ganho": 0},
                   {"remetente_desconhecido": True, **sinais_fracos})

h3 = rodar_hibrido("H3 · árvore suspeita, remetente conhecido",
                   {"num_links": 3, "termos_urgencia": 2, "termos_ganho": 2},
                   {"remetente_desconhecido": False, **sinais_fracos})

Árvore: {'num_links': 2, 'termos_urgencia': 3, 'termos_ganho': 0} -> suspeita_ml = sim
=== H1 · árvore suspeita + remetente desconhecido ===
Conclusão: características compatíveis com spam.
Ação recomendada: revisar a mensagem antes de decidir o que fazer.
Nenhuma mensagem foi movida ou apagada. A decisão final é humana.
Regras verificadas:
  R1 refutada: remetente_desconhecido = sim, link_encurtado = não
  R2 refutada: promessa_ganho_facil = não
  R4 confirmada: suspeita_ml = sim, remetente_desconhecido = sim
  R3 confirmada: spam = sim
Limitação: regras são hipóteses didáticas, sem validação empírica.

Árvore: {'num_links': 0, 'termos_urgencia': 0, 'termos_ganho': 0} -> suspeita_ml = nao
=== H2 · árvore não suspeita + remetente desconhecido ===
Conclusão: nenhuma regra confirmou spam.
Isso NÃO garante que a mensagem seja segura. Revise se necessário.
Regras verificadas:
  R1 refutada: remetente_desconhecido = sim, link_encurtado = não
  R2 refutada: promessa_ganho_facil = não
  R4 re

- **H1:** nenhum sinal informado pelo usuário confirma R1 ou R2, mas a árvore prevê suspeita, e **R4** confirma spam → revisão. Sem a árvore, o sistema original não chegaria a essa conclusão.
- **H2:** a árvore não vê suspeita, então R4 é refutada → nenhuma regra confirma.
- **H3:** a árvore vê suspeita, mas o remetente é conhecido, então R4 é refutada. A regra humana **limita** a previsão do modelo.

**Comparação** (passo 4): mesmos dados numéricos, mudando **só** o fato `remetente_desconhecido`.

In [12]:
atributos = {"num_links": 2, "termos_urgencia": 3, "termos_ganho": 0}
for desconhecido in (True, False):
    r = rodar_hibrido(f"remetente_desconhecido = {'sim' if desconhecido else 'não'}",
                      atributos, {"remetente_desconhecido": desconhecido, **sinais_fracos})
    print("-> revisão recomendada?", {True: "sim", False: "não", None: "inconclusivo"}[r.valor], "\n")

Árvore: {'num_links': 2, 'termos_urgencia': 3, 'termos_ganho': 0} -> suspeita_ml = sim
=== remetente_desconhecido = sim ===
Conclusão: características compatíveis com spam.
Ação recomendada: revisar a mensagem antes de decidir o que fazer.
Nenhuma mensagem foi movida ou apagada. A decisão final é humana.
Regras verificadas:
  R1 refutada: remetente_desconhecido = sim, link_encurtado = não
  R2 refutada: promessa_ganho_facil = não
  R4 confirmada: suspeita_ml = sim, remetente_desconhecido = sim
  R3 confirmada: spam = sim
Limitação: regras são hipóteses didáticas, sem validação empírica.

-> revisão recomendada? sim 

Árvore: {'num_links': 2, 'termos_urgencia': 3, 'termos_ganho': 0} -> suspeita_ml = sim
=== remetente_desconhecido = não ===
Conclusão: nenhuma regra confirmou spam.
Isso NÃO garante que a mensagem seja segura. Revise se necessário.
Regras verificadas:
  R1 refutada: remetente_desconhecido = não
  R2 refutada: promessa_ganho_facil = não
  R4 refutada: suspeita_ml = sim, rem

**Questões de fechamento**
- Que parte **aprende com os dados**? A **árvore de decisão** (Fase 1).
- Que parte **aplica as regras humanas**? O **motor de inferência** (Fase 2).
- Por que 100% num teste pequeno exige cautela? Porque são poucos casos, a avaliação é instável e, aqui, os dados são fictícios.

## 8. Ética e limitações

- **Privacidade:** só respostas manuais e atributos simples. O sistema não armazena mensagens e não envia nada a serviços externos.
- **Viés e falso positivo:** um aviso legítimo com link encurtado e remetente novo pode acionar R1. Por isso a saída é **recomendação**, não descarte.
- **Transparência:** o rastro mostra quais regras foram confirmadas, refutadas ou ficaram inconclusivas, e quais fatos foram usados (inclusive a previsão da árvore).
- **Supervisão humana:** o sistema **nunca move nem apaga** mensagens. Entre o descarte automático (A) e o alerta com revisão humana (B), adotamos **B**.
- **Limitações:** as regras são hipóteses didáticas, **sem validação** com mensagens reais. A árvore foi treinada com **dados fictícios**. Golpes sem esses sinais podem passar (falso negativo).
- **Uso de IA na elaboração:** ferramentas de IA generativa (ChatGPT e Claude) apoiaram código, testes, documentação e este notebook. A equipe revisou o conteúdo e responde pela entrega.

Relatório completo: [`docs/analise-etica.md`](https://github.com/Lucasdoreac/filtro-spam-lfa/blob/main/docs/analise-etica.md) · Material de consulta: [`docs/material-de-consulta.md`](https://github.com/Lucasdoreac/filtro-spam-lfa/blob/main/docs/material-de-consulta.md)

## Uso interativo (opcional)

Para responder às perguntas você mesmo, tire o `#` da linha abaixo e execute a célula.

In [13]:
# from cli import main; main()

## 9. Cola da prova

Resumo de consulta rápida da A1. Cada subseção abaixo pode ser recolhida pelo índice do Colab.

**Material:** github.com/Lucasdoreac/filtro-spam-lfa/blob/main/docs/material-de-consulta.md · **Colab:** colab.research.google.com/github/Lucasdoreac/filtro-spam-lfa/blob/main/notebook.ipynb · **Ética:** github.com/Lucasdoreac/filtro-spam-lfa/blob/main/docs/analise-etica.md


### O sistema

- **O que faz:** triagem de spam. Recebe 4 fatos, respondidos com s, n ou ?: remetente desconhecido, link duvidoso, ganho fácil e urgência. Objetivo: `revisao_recomendada`. **Nunca apaga mensagem.**
- **Regras:**
  - R1: remetente desconhecido **E** link duvidoso → spam
  - R2: ganho fácil **E** urgência → spam
  - R3: spam → revisão humana
- **Motor (encadeamento para trás):** parte do objetivo, procura as regras que o concluem, e as condições dessas regras viram subobjetivos. Só pergunta ao usuário o que nenhuma regra conclui.
- **Detalhes:** 3 valores (sim, não, não sei); premissa falsa encerra a regra sem perguntar o resto; cada fato é perguntado uma vez (`Motor.fatos`); ciclo → inconclusivo; conflito → ordem fixa (R1 antes de R2).

| Caso | Respostas | Rastro | Resultado |
|---|---|---|---|
| A | rem=s, link=s | R1 ✓ → R3 ✓ | revisão (só 2 perguntas) |
| B | rem=n, ganho=s, urg=s | R1 ✗ → R2 ✓ → R3 ✓ | revisão |
| C | rem=n, ganho=n | R1 ✗, R2 ✗, R3 ✗ | nenhuma confirmou (**não** significa seguro) |
| D | rem=?, link=s, ganho=n | R1 inconclusiva, R2 ✗, R3 inconclusiva | inconclusivo |


### Teoria rápida

- **Para frente × para trás:** dados → conclusões × hipótese → evidências. Usamos para trás: objetivo único e menos perguntas.
- **Informação ausente:** "?" não é "não"; ausência de evidência não é negação.
- **Fator de certeza (MYCIN):** CF = CF1 + CF2 × (1 − CF1). Não usamos.
- **Como?** "veio de R3, apoiada por R1" (rastro). **Por quê?** "pergunto o link porque R1 precisa dele" (não é interativo no programa).
- **Componentes:** base de conhecimento, memória de trabalho, motor, interface, explicação, aquisição.
- **Papéis:** especialista do domínio, engenheiro do conhecimento, usuário.
- **Clássicos:** DENDRAL (química), MYCIN (infecções). **Ferramentas:** CLIPS (regras); Prolog `avo(X,Z) :- pai(X,Y), pai(Y,Z).`
- **Testes:** positivo, negativo e informação ausente; 16 testes, todos passando no GitHub Actions. Verificação (lógica certa?) feita; validação com mensagens reais **não** feita.


### Aprendizado de máquina (no notebook)

- X = atributos, y = rótulo. `train_test_split(test_size=0.25, stratify=y, random_state=42)`; `DecisionTreeClassifier(max_depth=2)` → `fit` → `predict`.
- Matriz de confusão e acurácia: 83% em 6 casos, com dados **fictícios**. 100% num teste pequeno exige cautela.
- **R4:** `suspeita_ml` = sim **E** remetente desconhecido → spam. A previsão do modelo vira um **fato** para as regras.
- Que parte aprende com os dados? A árvore. Que parte aplica as regras humanas? O motor.


### Ética

- **Vídeo (IEA-USP, 2h01):** **00:06:18** ética não é só lista de conformidade; contam instituições, incentivos e responsabilidade. **01:35:50** IA vendida como "neutra" ou "solução mágica" não é neutra nem nos dados nem nos vieses.
- **Artigo (Lacerda Júnior, 2026):** nem proibir nem liberar sem critério; políticas graduadas, com declaração de uso e responsabilização humana; matriz definir, ensinar, redesenhar, proteger, acompanhar.
- **Dilema:** A = descartar automaticamente; B = alertar e pedir revisão humana. Escolhemos **B**: apagar mensagem legítima é irreversível. O trecho de 01:35:50 apoia essa escolha.
- **IA usada no trabalho:** ChatGPT e Claude, conferidos pela equipe.


### Pontos fracos (se perguntarem)

Sem fator de certeza, "Por quê?" não interativo, só 3 regras, ML com dados fictícios, sem validação com mensagens reais.


### Por que só 3 regras? (outros grupos têm 12, 52…)

- Domínios maiores pedem mais regras: golpe bancário (G6, 52), bolsa (G25, 12), solo (G22, 12).
- Muitos criam regra só para "?" (G25 R03, G23 R09, G15 R5). Nós tratamos o "?" no **motor**, com 3 valores.
- Muitos separam "o que é" de "o que fazer" (G23: R01 detecta, R07 age). Nós temos uma ação só (R3).
- Às vezes "regra" é só uma checagem (G18: R1–R8 "registrar motivo"), o que infla o número.
- Exigência: a ficha pede pelo menos **2 regras**, e temos 3 (+R4 no notebook). O slide 40 pedia 5 fatos e 4 regras; temos 4 fatos e 4 regras.
- O que pesa: regras coerentes e verificadas, sem contradição, ciclo ou regra morta (slide 36). Os 16 testes cobrem todas.
- **Resposta pronta:** "O domínio é delimitado de propósito: quatro sinais e uma única ação segura, a revisão humana. Duas regras detectam spam por caminhos diferentes e uma decide a ação. A incerteza fica no motor, não em regras extras. Mais regras sem validação com mensagens reais só aumentariam os falsos positivos."


### Resto da disciplina (conteúdo padrão de IA e LFA; não conferido com os slides)

**Agentes:** percebem por sensores e agem por atuadores. PEAS = desempenho, ambiente, atuadores, sensores. Tipos: reativo simples (condição → ação), reativo com modelo (estado interno), baseado em objetivo, em utilidade e com aprendizado. Ambientes: observável ou parcial, determinístico ou estocástico, episódico ou sequencial, estático ou dinâmico, discreto ou contínuo, um ou vários agentes.

**Busca:** largura (completa, muita memória); profundidade (pouca memória, pode não terminar); custo uniforme. Gulosa usa h(n). **A\***: f = g + h, ótima se h é **admissível** (nunca superestima).

**ML:** supervisionado (com rótulo: classificação ou regressão); não supervisionado (sem rótulo, ex. k-means); por reforço (recompensa). **Overfitting:** decora o treino. Precisão = VP/(VP+FP); recall = VP/(VP+FN).

**LFA, básico:** alfabeto Σ (símbolos); cadeia; ε = cadeia vazia; linguagem = conjunto de cadeias; Σ\* = todas as cadeias.

**AFD** (Q, Σ, δ, q0, F): uma transição por símbolo; aceita se termina em F. **AFN:** várias transições ou ε; vira AFD pela construção de subconjuntos.

**Expressão regular:** `a|b` (união), `ab` (concatenação), `a*` (fecho de Kleene). Descreve as linguagens regulares, as mesmas dos autômatos finitos. **Bombeamento:** prova que uma linguagem **não** é regular (ex.: aⁿbⁿ).

| Tipo | Gramática | Reconhecedor | Exemplo |
|---|---|---|---|
| 3 | regular | autômato finito | a\*b |
| 2 | livre de contexto | autômato de pilha | aⁿbⁿ, parênteses |
| 1 | sensível ao contexto | linearmente limitado | aⁿbⁿcⁿ |
| 0 | irrestrita | máquina de Turing | rec. enumerável |

**Gramática livre de contexto:** regras A → α; derivação, árvore, **ambiguidade** (duas árvores para a mesma cadeia). **Máquina de Turing:** fita infinita, cabeça que lê e escreve; existem problemas **indecidíveis** (problema da parada).

**LFA dentro da IA:** AFD e expressão regular extraem fatos do texto (ex.: o G23 detecta link ou pix). No nosso projeto, poderiam responder sozinhos "tem link encurtado?".


### Aula 12, outros pontos

- **SE:** domínio restrito; separa conhecimento (base) de raciocínio (motor). **Tarefas:** diagnóstico, interpretação, configuração e planejamento. **Domínio delimitado:** entradas e saídas reconhecíveis, especialistas e casos para conferir, critério de sucesso.
- **Ciclo de inferência:** achar regras satisfeitas → escolher uma → executar → atualizar fatos → repetir. **Parada:** objetivo alcançado, nada novo ou limite de execução.
- **Conflito:** prioridade, especificidade (mais condições vence) ou recência; a estratégia deve ser explícita e previsível.
- **Aquisição:** entrevistas, observação de casos, documentos e protocolos, comparação entre especialistas. **Formalização:** delimitar termos e valores, transformar relatos em condições verificáveis, registrar exceções.
- **Construção:** adquirir e formalizar → implementar regras, interação e explicação → testar com casos e revisar com especialistas.
- **Benefícios:** consultas recorrentes, critérios padronizados, justificativas registradas, treinamento. **Limitações:** conhecimento tácito difícil de formalizar, regras desatualizadas, falha fora do domínio.
- **Responsabilidade:** o usuário conhece o alcance; a base registra origem, versão e responsáveis; o que está fora do escopo vai para um humano.
- **No nosso código:** base = `base.py`; memória = `Motor.fatos`; motor = `motor.py`; interface e explicação = `cli.py` (`explicar`).


### Notebook: casos híbridos

- **H1:** a árvore prevê suspeita + remetente desconhecido → R1 e R2 refutadas, **R4 confirma** → revisão. Sem a árvore, o sistema não concluiria isso.
- **H2:** a árvore não prevê suspeita → R4 refutada → nenhuma regra confirma.
- **H3:** a árvore prevê suspeita, mas o remetente é conhecido → R4 refutada: a regra humana limita o modelo.
- A árvore aprendeu a olhar só os **termos de urgência**; errou 1 de 6 casos de teste.


### Ética: 5 eixos (situação → medida)

| Eixo | Situação | Medida |
|---|---|---|
| Privacidade | a mensagem pode revelar dados pessoais | só 4 respostas; nada armazenado |
| Viés | aviso legítimo com link curto aciona R1 | recomenda, não descarta; teste de falso positivo |
| Transparência | alerta sem motivo claro | rastro das regras + aviso de que não foram validadas |
| Supervisão | mensagem legítima marcada | não move nem apaga; decisão humana |
| Impacto | erra nos dois sentidos | limites explícitos; testar antes de uso real |


### Perguntas prováveis

- **Conhecimento difícil de virar regra?** "Link duvidoso" e "urgência" exigem julgamento, por isso a pessoa responde.
- **Quando suspender a recomendação?** Quando há "?" e fora do domínio; por isso nunca há descarte automático.
- **Evidência de que a base precisa de revisão?** Muitos falsos positivos ou negativos numa amostra real e autorizada.
- **Por que "não" ≠ seguro?** Só 4 sinais são verificados: golpe sem esses sinais passa (falso negativo).
- **Ciclo de regras?** O motor devolve inconclusivo (`test_ciclo_termina_como_inconclusivo`).
- **Como rodar?** `python cli.py`; `python -m unittest discover -v`; ou o notebook no Colab.